In [1]:
from pathlib import Path

import altair as alt
import polars as pl

from src.utils.date import monthYear

DATA_DIR = Path("../../data/measurements")

df = (
    pl.read_parquet(DATA_DIR / "ech_public_name.parquet")
    .group_by("test_date", "ech_public_name")
    .agg(pl.col("record_count").sum().alias("record_count"))
)

In [2]:
chart = (
    alt.Chart(df)
    .mark_circle(size=100)
    .encode(
        x=alt.X(
            "test_date:T",
            # title="Date",
            title=None,
            scale=alt.Scale(padding=10),
            axis=alt.Axis(format=monthYear),
        ),
        y=alt.Y(
            "ech_public_name:N",
            title="ECH Public Name",
        ),
        color=alt.Color(
            "record_count:Q",
            scale=alt.Scale(scheme="turbo", type="log"),
            title="Domains per ECH Public Name",
        ),
        tooltip=["ech_public_name", "test_date:O", "record_count:Q"],
    )
    .properties(
        height=250,
    )
    .configure_legend(orient="none", direction="horizontal", legendX=-135, legendY=285, gradientLength=175)
)

# Save the chart as a PDF
chart.save("../../plots/public_name_count.pdf")
chart.show()

alt.Chart(...)